## Import Libraries

In [1]:
import numpy as np
import pandas as pd

from gensim.models import Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.metrics.pairwise import cosine_similarity

## Example Dataset

In [2]:
documents = [
    "I love natural language processing",
    "NLP is an interesting field",
    "Machine learning helps computers understand language",
    "Deep learning improves NLP models",
    "Word embeddings capture semantic meaning"
]

## Preprocessing 

Tokenization

In [3]:
tokenized_docs = [doc.lower().split() for doc in documents]

## PART 1 — Averaging Word Embeddings

### Train Word2Vec

In [4]:
w2v_model = Word2Vec(
    sentences=tokenized_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4
)

### Create Document Vectors (Averaging)

In [5]:
def document_vector(doc):
    vectors = [w2v_model.wv[word] for word in doc if word in w2v_model.wv]
    return np.mean(vectors, axis=0)

In [7]:
doc_vectors = [document_vector(doc) for doc in tokenized_docs]

## Similarity Between Documents

In [8]:
cosine_similarity([doc_vectors[0]], [doc_vectors[1]])

array([[0.13470294]], dtype=float32)

In [9]:
cosine_similarity(doc_vectors)

array([[ 1.        ,  0.13470292,  0.11312919, -0.17265658, -0.29072663],
       [ 0.13470292,  1.        ,  0.11188219,  0.24819481, -0.15990706],
       [ 0.11312919,  0.11188219,  1.0000001 ,  0.31214115,  0.07862736],
       [-0.17265658,  0.24819481,  0.31214115,  1.0000001 , -0.00226431],
       [-0.29072663, -0.15990706,  0.07862736, -0.00226431,  0.99999994]],
      dtype=float32)

## PART 2 — Doc2Vec

### Prepare Tagged Data

In [10]:
tagged_docs = [
    TaggedDocument(words=doc, tags=[str(i)])
    for i, doc in enumerate(tokenized_docs)
]

### Train Doc2Vec Model

In [11]:
d2v_model = Doc2Vec(
    vector_size=50,
    window=3,
    min_count=1,
    workers=4,
    epochs=100
)

d2v_model.build_vocab(tagged_docs)
d2v_model.train(tagged_docs, total_examples=d2v_model.corpus_count, epochs=d2v_model.epochs)

### Get Document Vectors

In [12]:
d2v_model.dv["0"]

array([-0.01434276, -0.01357013, -0.02143164,  0.0170658 ,  0.00717907,
        0.00105537, -0.02036209, -0.00971568, -0.02198211,  0.00571919,
        0.00669923,  0.01164985, -0.00864831, -0.0092744 , -0.00441936,
       -0.02037852,  0.00331844,  0.01969044, -0.0223714 , -0.00785271,
       -0.00876166,  0.00576281, -0.01249732,  0.00769967,  0.01060306,
       -0.01653014, -0.01913292, -0.0225093 ,  0.00994759, -0.02085785,
        0.01485343,  0.01471956, -0.01540919, -0.01064926, -0.00414996,
        0.00500427, -0.00364488, -0.01959699, -0.00754634,  0.00264062,
       -0.00355073, -0.01295772,  0.01006825, -0.02006942,  0.0049534 ,
       -0.00800286,  0.0026224 , -0.00325552,  0.01260967, -0.01858035],
      dtype=float32)

In [14]:
doc_vectors_d2v = [d2v_model.dv[str(i)] for i in range(len(documents))]

In [16]:
doc_vectors_d2v

[array([-0.01434276, -0.01357013, -0.02143164,  0.0170658 ,  0.00717907,
         0.00105537, -0.02036209, -0.00971568, -0.02198211,  0.00571919,
         0.00669923,  0.01164985, -0.00864831, -0.0092744 , -0.00441936,
        -0.02037852,  0.00331844,  0.01969044, -0.0223714 , -0.00785271,
        -0.00876166,  0.00576281, -0.01249732,  0.00769967,  0.01060306,
        -0.01653014, -0.01913292, -0.0225093 ,  0.00994759, -0.02085785,
         0.01485343,  0.01471956, -0.01540919, -0.01064926, -0.00414996,
         0.00500427, -0.00364488, -0.01959699, -0.00754634,  0.00264062,
        -0.00355073, -0.01295772,  0.01006825, -0.02006942,  0.0049534 ,
        -0.00800286,  0.0026224 , -0.00325552,  0.01260967, -0.01858035],
       dtype=float32),
 array([-0.0082421 , -0.00141557, -0.01458156, -0.01395889, -0.00430343,
         0.01862585, -0.00257719,  0.00845539, -0.01370795,  0.01969297,
         0.00702325,  0.02104317,  0.00929976, -0.01153017,  0.00682382,
        -0.01174626,  0.010

### Document Similarity (Doc2Vec)

In [15]:
cosine_similarity([doc_vectors_d2v[0]], [doc_vectors_d2v[1]])

array([[0.36961678]], dtype=float32)

## Inference on New Document

In [17]:
new_doc = "I enjoy learning NLP"
new_tokens = new_doc.lower().split()

new_vector = d2v_model.infer_vector(new_tokens)

In [18]:
cosine_similarity([new_vector], doc_vectors_d2v)

array([[0.44837078, 0.39449596, 0.2949529 , 0.28347826, 0.08514404]],
      dtype=float32)

## Comparison 

Averaging Word2Vec:

pros:
- Simple
- Fast
  
cons:
- Loses word order

Doc2Vec:

pros:
- Learns document context
- Better representation

cons:
- Requires training